# DO Card Classifier Training

Train a simple full-card classifier using the data produced by the DO notebooks: reference card crops, augmented card crops, and labeled crops from augmented scenes. The notebook saves a single classifier that predicts labels such as `r_5`, `wild`, or `draw_4` directly.

In [ ]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd() / "project" / "notebooks" / "do" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate do/src directory for imports.")


In [ ]:
from src.train_classifier import (
    TrainPipelineConfig,
    initialize_training_pipeline,
    plot_sample_preview,
    plot_wrong_predictions,
    run_feature_extraction,
    run_training,
    run_validation_diagnostics,
    save_training_artifacts,
)


## Configuration

In [ ]:
CFG = TrainPipelineConfig()
CFG


## Initialize Pipeline

Locate paths, load reference / augmented-card / scene samples, and assert no test data leaks in (R3).

In [ ]:
state = initialize_training_pipeline(CFG)
sorted(state.keys())


## Sample Preview

Quick visual check of the mixed training set.

In [ ]:
plot_sample_preview(state)


## Feature Extraction

Global HSV histogram + global/center HOG + global/center Fourier descriptors.

In [ ]:
state = run_feature_extraction(state)
state["X"].shape, len(state["label_encoder"].classes_)


## Train

ExtraTrees classifier with source-aware sample weighting and a tiny hyperparameter sweep.

In [ ]:
state = run_training(state)
state["best_params"], state["val_acc"]


## Save Artifacts

In [ ]:
state = save_training_artifacts(state)


## Validation Diagnostics

In [ ]:
state = run_validation_diagnostics(state)


## Wrong Predictions

In [ ]:
plot_wrong_predictions(state)
